In [1]:
# pip install groq
import os
from dataclasses import dataclass
from groq import Groq
from dotenv import load_dotenv
import json

In [2]:
# pip install soundfile
from pathlib import Path
import soundfile as sf

In [3]:
SUPPORTED_EXTS = {".wav", ".mp3"}
MAX_SIZE_MB = 100


@dataclass
class AudioInput:
    path: Path
    format: str
    size_mb: float
    duration_sec: float
    sample_rate: int
    channels: int


def load_audio(file_path: str) -> AudioInput:
    """Validate an audio file and return its basic metadata."""
    path = Path(file_path)

    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    ext = path.suffix.lower()
    if ext not in SUPPORTED_EXTS:
        raise ValueError(f"Unsupported format '{ext}'. Use one of {SUPPORTED_EXTS}")

    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb == 0:
        raise ValueError("File is empty")
    if size_mb > MAX_SIZE_MB:
        raise ValueError(f"File too large: {size_mb:.1f} MB (max {MAX_SIZE_MB} MB)")

    try:
        info = sf.info(str(path))  # reads header only, fast
    except Exception as e:
        raise ValueError(f"Could not read audio file (corrupt or unsupported): {e}")

    return AudioInput(
        path=path,
        format=ext.lstrip("."),
        size_mb=round(size_mb, 2),
        duration_sec=round(info.duration, 2),
        sample_rate=info.samplerate,
        channels=info.channels,
    )

In [4]:

load_dotenv()  # reads .env into environment variables
if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError("GROQ_API_KEY not found. Add it to your .env file.")


client = Groq(max_retries=3, timeout=120)

MODEL = "whisper-large-v3-turbo"  # fast; use "whisper-large-v3" for max accuracy
API_MAX_MB = 25  # free-tier upload limit


@dataclass
class Segment:
    start: float
    end: float
    text: str


@dataclass
class Transcript:
    text: str
    language: str
    segments: list[Segment]


def _get(obj, key):
    """Segments may come back as dicts or objects depending on SDK version."""
    return obj[key] if isinstance(obj, dict) else getattr(obj, key)


def transcribe(audio: AudioInput, language: str | None = None) -> Transcript:
    """Transcribe via Groq's Whisper API. Language is auto-detected if not given (e.g. 'en')."""
    if audio.size_mb > API_MAX_MB:
        raise ValueError(f"File is {audio.size_mb} MB; limit is {API_MAX_MB} MB. Compress or split it.")

    with open(audio.path, "rb") as f:
        params = dict(
            file=(audio.path.name, f.read()),
            model=MODEL,
            response_format="verbose_json",
            timestamp_granularities=["segment"],
        )
        if language:
            params["language"] = language
        resp = client.audio.transcriptions.create(**params)

    segments = [
        Segment(round(_get(s, "start"), 2), round(_get(s, "end"), 2), _get(s, "text").strip())
        for s in (getattr(resp, "segments", None) or [])
    ]
 
    return Transcript(
        text=resp.text.strip(),
        language=getattr(resp, "language", None) or language or "unknown",
        segments=segments,
    )

In [6]:

def fmt_time(seconds: float) -> str:
    """Seconds -> HH:MM:SS.mmm"""
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02}:{m:02}:{s:02}.{ms:03}"


def timestamped_text(t: Transcript) -> str:
    """Human-readable: one line per segment."""
    return "\n".join(
        f"[{fmt_time(s.start)} --> {fmt_time(s.end)}] {s.text}" for s in t.segments
    )


def to_dict(t: Transcript) -> dict:
    """JSON-friendly output for downstream use."""
    return {
        "language": t.language,
        "text": t.text,
        "segments": [{"start": s.start, "end": s.end, "text": s.text} for s in t.segments],
    }


# Usage
audio = load_audio("sample-speech-1m.wav")
result = transcribe(audio)

print(timestamped_text(result))

with open("transcript.json", "w", encoding="utf-8") as f:
    json.dump(to_dict(result), f, ensure_ascii=False, indent=2)

[00:00:00.000 --> 00:00:05.800] Welcome to SampleLid.com, a free online resource for downloading sample files in a wide variety
[00:00:05.800 --> 00:00:07.000] of digital formats.
[00:00:07.360 --> 00:00:12.540] Whether you are a software developer testing file upload functionality, a quality assurance
[00:00:12.540 --> 00:00:17.840] engineer validating media players, a student learning about digital formats, or simply
[00:00:17.840 --> 00:00:23.400] someone who needs a quick test file, SampleLid provides ready-to-use files that you can download
[00:00:23.400 --> 00:00:25.660] instantly, completely free of charge.
[00:00:25.660 --> 00:00:30.180] In this recording, we will walk you through every aspect of the Samplilib platform,
[00:00:30.560 --> 00:00:34.720] exploring the formats we offer, the technical details behind each one,
[00:00:34.940 --> 00:00:38.720] and the many ways these test files can be used in your projects and workflows.
[00:00:39.200 --> 00:00:44.200] Let us begin wit